# FreshMart Lab 3: Train and Track Models with MLflow
**Microsoft Fabric Data Science**

ทำนาย **Churn** (classification) แล้วเทียบ Decision Tree กับ Random Forest

**AUC** = คะแนนแยกกลุ่ม Churn (ใกล้ 1 ดีกว่า) — เก็บตัวชนะเป็น `freshmart-churn-model`


In [ ]:
import json
from pathlib import Path
import pandas as pd

from pathlib import Path
import pandas as pd

def _first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None

def load_csv(file_name: str) -> pd.DataFrame:
    found = _first_existing([
        f"/lakehouse/default/Files/raw/{file_name}",
        f"Files/raw/{file_name}",
        f"../data/{file_name}",
        f"labs/data/{file_name}",
        file_name,
    ])
    if found is None:
        raise FileNotFoundError(f"Cannot find {file_name}. Upload it to Files/raw or place it under labs/data.")
    print(f"Loaded CSV: {found}")
    return pd.read_csv(found)

def load_table_or_csv(table_name: str, file_name: str) -> pd.DataFrame:
    try:
        frame = spark.read.table(table_name).toPandas()
        print(f"Loaded Spark table {table_name}: {len(frame):,} rows")
        return frame
    except Exception as exc:
        print(f"Spark table '{table_name}' unavailable ({exc}). Falling back to CSV.")
        return load_csv(file_name)


In [ ]:
from dataclasses import asdict, dataclass
from typing import Any
import logging

logger = logging.getLogger(__name__)

ID_COLUMN = 'CustomerID'
TARGET_COLUMN = 'Churn'
CATEGORICAL_COLUMNS = ('MembershipTier', 'Gender')
IMPUTE_MEDIAN_COLUMNS = ('Age',)
SCALE_COLUMNS = ('MonetaryTotal', 'AvgBasketSize', 'RecencyDays', 'TenureMonths')
DUMMY_COLUMNS = ('MembershipTier_Bronze', 'MembershipTier_Gold', 'MembershipTier_Platinum', 'MembershipTier_Silver', 'Gender_F', 'Gender_M', 'Gender_Other')
PASSTHROUGH_COLUMNS = ('Frequency', 'ComplaintCount')
FEATURE_COLUMNS = ('Age', 'TenureMonths', 'RecencyDays', 'Frequency', 'MonetaryTotal', 'AvgBasketSize', 'ComplaintCount', 'MembershipTier_Bronze', 'MembershipTier_Gold', 'MembershipTier_Platinum', 'MembershipTier_Silver', 'Gender_F', 'Gender_M', 'Gender_Other')
REQUIRED_RAW_COLUMNS = ('CustomerID', 'Age', 'Gender', 'MembershipTier', 'TenureMonths', 'RecencyDays', 'Frequency', 'MonetaryTotal', 'AvgBasketSize', 'ComplaintCount')

@dataclass(frozen=True)
class FeatureParams:
    """Fitted preprocessing parameters reused at scoring time.

    Attributes:
        age_median: Median Age computed from the training customers.
        scale_mins: Per-column minimum used for min-max scaling.
        scale_maxs: Per-column maximum used for min-max scaling.
        dummy_columns: One-hot columns the model expects, in order.
        feature_columns: Final model input columns, in order.
    """

    age_median: float
    scale_mins: dict[str, float]
    scale_maxs: dict[str, float]
    dummy_columns: list[str]
    feature_columns: list[str]

    def to_dict(self) -> dict[str, Any]:
        """Serialize parameters to a JSON-friendly dictionary."""
        return asdict(self)

    @classmethod
    def from_dict(cls, payload: dict[str, Any]) -> FeatureParams:
        """Create parameters from a dictionary.

        Args:
            payload: Mapping produced by ``to_dict``.

        Returns:
            Validated ``FeatureParams``.

        Raises:
            ValueError: If required keys are missing.
        """
        required = {
            "age_median",
            "scale_mins",
            "scale_maxs",
            "dummy_columns",
            "feature_columns",
        }
        missing = required - set(payload)
        if missing:
            raise ValueError(f"FeatureParams missing keys: {sorted(missing)}")
        return cls(
            age_median=float(payload["age_median"]),
            scale_mins={k: float(v) for k, v in payload["scale_mins"].items()},
            scale_maxs={k: float(v) for k, v in payload["scale_maxs"].items()},
            dummy_columns=list(payload["dummy_columns"]),
            feature_columns=list(payload["feature_columns"]),
        )


def validate_raw_customers(df: pd.DataFrame, *, require_target: bool = True) -> pd.DataFrame:
    """Validate and normalize a raw FreshMart customer frame.

    Args:
        df: Raw customer records from CSV or ``bronze_customers``.
        require_target: When True, require the ``Churn`` column.

    Returns:
        Copy with numeric columns coerced.

    Raises:
        ValueError: If required columns are missing.
    """
    required = REQUIRED_RAW_COLUMNS + ((TARGET_COLUMN,) if require_target else ())
    _require_columns(df, required, frame_name="customer frame")
    numeric_cols = (
        "Age",
        "TenureMonths",
        "RecencyDays",
        "Frequency",
        "MonetaryTotal",
        "AvgBasketSize",
        "ComplaintCount",
    )
    if require_target:
        numeric_cols = numeric_cols + (TARGET_COLUMN,)
    return _numeric_copy(df, numeric_cols)


def _require_columns(df: pd.DataFrame, columns: tuple[str, ...], *, frame_name: str) -> None:
    """Raise if expected columns are missing."""
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"{frame_name} missing required columns: {missing}")


def _numeric_copy(df: pd.DataFrame, columns: tuple[str, ...]) -> pd.DataFrame:
    """Return a copy with selected columns coerced to numeric."""
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def fit_preprocessor(df_raw: pd.DataFrame) -> FeatureParams:
    """Fit imputation and scaling parameters on training customers.

    Args:
        df_raw: Raw customer DataFrame including ``Churn``.

    Returns:
        Fitted parameters that must be reused for scoring.
    """
    df = validate_raw_customers(df_raw, require_target=True)
    age_median = float(df["Age"].median())
    if pd.isna(age_median):
        raise ValueError("Cannot fit preprocessor: Age median is NaN")

    scale_mins: dict[str, float] = {}
    scale_maxs: dict[str, float] = {}
    for col in SCALE_COLUMNS:
        col_min = float(df[col].min())
        col_max = float(df[col].max())
        if col_max <= col_min:
            raise ValueError(f"Cannot scale {col}: min={col_min}, max={col_max}")
        scale_mins[col] = col_min
        scale_maxs[col] = col_max

    params = FeatureParams(
        age_median=age_median,
        scale_mins=scale_mins,
        scale_maxs=scale_maxs,
        dummy_columns=list(DUMMY_COLUMNS),
        feature_columns=list(FEATURE_COLUMNS),
    )
    logger.info(
        "Fitted feature params: age_median=%.1f, scale_columns=%s",
        params.age_median,
        list(SCALE_COLUMNS),
    )
    return params


def _one_hot_categories(df: pd.DataFrame) -> pd.DataFrame:
    """One-hot encode membership and gender, keeping the expected columns."""
    encoded = pd.get_dummies(df, columns=list(CATEGORICAL_COLUMNS), drop_first=False)
    for col in DUMMY_COLUMNS:
        if col not in encoded.columns:
            encoded[col] = 0
    return encoded


def _min_max_scale(series: pd.Series, col_min: float, col_max: float) -> pd.Series:
    """Scale a series to [0, 1] using fitted min/max."""
    return (series - col_min) / (col_max - col_min + 1e-6)


def transform_customers(
    df_raw: pd.DataFrame,
    params: FeatureParams,
    *,
    require_target: bool = False,
) -> pd.DataFrame:
    """Apply the fitted FreshMart preprocessing contract.

    Args:
        df_raw: Raw customer records (training or scoring batch).
        params: Parameters from ``fit_preprocessor``.
        require_target: When True, keep and validate ``Churn``.

    Returns:
        Feature frame with ``CustomerID``, model columns, and optional ``Churn``.
    """
    df = validate_raw_customers(df_raw, require_target=require_target)
    work = df.copy()
    work["Age"] = work["Age"].fillna(params.age_median)

    work = _one_hot_categories(work)
    for col in SCALE_COLUMNS:
        work[col] = _min_max_scale(
            work[col],
            params.scale_mins[col],
            params.scale_maxs[col],
        )

    bool_cols = work.select_dtypes(include="bool").columns
    work[bool_cols] = work[bool_cols].astype(int)

    ordered = [ID_COLUMN, *params.feature_columns]
    if require_target or TARGET_COLUMN in work.columns:
        ordered.append(TARGET_COLUMN)
    missing = [col for col in ordered if col not in work.columns]
    if missing:
        raise ValueError(f"Transformed frame missing columns: {missing}")

    result = work[ordered].copy()
    feature_frame = result[list(params.feature_columns)]
    if feature_frame.isna().any().any():
        bad = feature_frame.columns[feature_frame.isna().any()].tolist()
        raise ValueError(f"NaN remaining in feature columns: {bad}")
    return result


def model_matrix(df_features: pd.DataFrame, params: FeatureParams) -> pd.DataFrame:
    """Return the model input matrix in signature order.

    Args:
        df_features: Output of ``transform_customers``.
        params: Fitted parameters.

    Returns:
        DataFrame with only model feature columns.
    """
    _require_columns(df_features, tuple(params.feature_columns), frame_name="feature frame")
    return df_features[list(params.feature_columns)].astype(float)


def save_feature_params(params: FeatureParams, path: str | Path) -> Path:
    """Write feature parameters to a JSON file.

    Args:
        params: Fitted parameters.
        path: Destination JSON path.

    Returns:
        Resolved output path.
    """
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(params.to_dict(), indent=2), encoding="utf-8")
    logger.info("Saved feature params to %s", out)
    return out


def load_feature_params(path: str | Path) -> FeatureParams:
    """Load feature parameters from a JSON file.

    Args:
        path: JSON path written by ``save_feature_params``.

    Returns:
        Fitted parameters.

    Raises:
        FileNotFoundError: If the file does not exist.
    """
    src = Path(path)
    if not src.exists():
        raise FileNotFoundError(f"Feature params not found: {src}")
    payload = json.loads(src.read_text(encoding="utf-8"))
    return FeatureParams.from_dict(payload)


### ขั้นตอนที่ 1: โหลด Silver features


In [ ]:
df_features = load_table_or_csv("silver_customer_features", ".local/silver_customer_features.csv")
if "MembershipTier_Bronze" not in df_features.columns:
    raw = load_table_or_csv("bronze_customers", "freshmart_customers.csv")
    params = fit_preprocessor(raw)
    df_features = transform_customers(raw, params, require_target=True)
else:
    params_path = _first_existing([
        "Files/params/feature_params.json",
        "/lakehouse/default/Files/params/feature_params.json",
        "labs/data/.local/feature_params.json",
    ])
    if params_path:
        params = load_feature_params(params_path)
    else:
        raw = load_table_or_csv("bronze_customers", "freshmart_customers.csv")
        params = fit_preprocessor(raw)

X = model_matrix(df_features, params)
y = df_features["Churn"].astype(int)
print(f"Features ({len(X.columns)}):", list(X.columns))
print(f"Dataset shape: X={X.shape}, y={y.shape}")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train size: {len(X_train):,}, Test size: {len(X_test):,}")


### ขั้นตอนที่ 2: MLflow Experiment


In [ ]:
mlflow_ready = True
try:
    import mlflow
    import mlflow.sklearn
    from mlflow.models.signature import infer_signature
    experiment_name = "freshmart-churn-prediction"
    mlflow.set_experiment(experiment_name)
    print(f"MLflow Experiment set to: '{experiment_name}'")
except Exception as exc:
    mlflow_ready = False
    infer_signature = None
    print(f"MLflow unavailable ({exc}). Training will still run locally.")


### ขั้นตอนที่ 3: Baseline Decision Tree


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate(model, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1_score": f1_score(y_test, y_pred),
        "test_roc_auc": roc_auc_score(y_test, y_prob),
    }
    print(f"[{name}] F1={metrics['test_f1_score']:.4f} Recall={metrics['test_recall']:.4f} AUC={metrics['test_roc_auc']:.4f}")
    return metrics

dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
if mlflow_ready:
    with mlflow.start_run(run_name="Run_01_DecisionTree_Baseline"):
        mlflow.autolog(log_models=False)
        dt_model.fit(X_train, y_train)
        metrics = evaluate(dt_model, "Run 1 Decision Tree")
        for key, value in metrics.items():
            mlflow.log_metric(key, value)
        signature = infer_signature(X_train, dt_model.predict(X_train))
        mlflow.sklearn.log_model(dt_model, "model", signature=signature)
else:
    dt_model.fit(X_train, y_train)
    evaluate(dt_model, "Run 1 Decision Tree")


### ขั้นตอนที่ 4: Champion Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
if mlflow_ready:
    with mlflow.start_run(run_name="Run_02_RandomForest_Champion"):
        mlflow.autolog(log_models=False)
        rf_model.fit(X_train, y_train)
        metrics = evaluate(rf_model, "Run 2 Random Forest")
        for key, value in metrics.items():
            mlflow.log_metric(key, value)
        cm = confusion_matrix(y_test, rf_model.predict(X_test))
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=["Stay (0)", "Churn (1)"], yticklabels=["Stay (0)", "Churn (1)"])
        plt.title("FreshMart Churn Confusion Matrix")
        plt.ylabel("Actual")
        plt.xlabel("Predicted")
        plt.tight_layout()
        plt.savefig("confusion_matrix.png")
        plt.show()
        mlflow.log_artifact("confusion_matrix.png")
        signature = infer_signature(X_train, rf_model.predict(X_train))
        mlflow.sklearn.log_model(rf_model, "model", signature=signature)
else:
    rf_model.fit(X_train, y_train)
    evaluate(rf_model, "Run 2 Random Forest")


### ขั้นตอนที่ 5: ลงทะเบียน Champion


In [ ]:
dt_auc = roc_auc_score(y_test, dt_model.predict_proba(X_test)[:, 1])
rf_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])
if rf_auc <= dt_auc:
    raise AssertionError(
        f"Random Forest (AUC={rf_auc:.4f}) ควรชนะ Decision Tree (AUC={dt_auc:.4f}) — ตรวจลำดับฟีเจอร์/สเกล"
    )
print(f"Champion AUC: {rf_auc:.4f}")

if mlflow_ready:
    exp = mlflow.get_experiment_by_name(experiment_name)
    runs = mlflow.search_runs(exp.experiment_id, order_by=["metrics.test_roc_auc DESC"], max_results=1)
    champion_run_id = runs.iloc[0]["run_id"]
    model_uri = f"runs:/{champion_run_id}/model"
    mv = mlflow.register_model(model_uri, "freshmart-churn-model")
    print(f"Registered {mv.name} version {mv.version}")
else:
    print("Skip registry (local mode). Champion model remains in-memory as rf_model.")
print("Lab 3 verification passed")
